In [5]:
!pip install trl accelerate transformers torch datasets requests -q
!pip install -q peft

In [6]:
import os
os.environ["ENV_URL"] = "https://ishikamahadar-resume-env.hf.space"
os.environ["MODEL_NAME"] = "Qwen/Qwen2.5-1.5B-Instruct"

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [12]:
"""
train_grpo.py — GRPO training for the Hiring Fleet environment.
"""

import os, json, re, random, time, traceback
import torch
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ─────────────────────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────────────────────
ENV_URL    = os.getenv("ENV_URL",    "https://ishikamahadar-resume-env.hf.space")
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
HF_TOKEN   = os.getenv("HF_TOKEN",  "")

N_COLLECT_EPISODES = 36
GROUP_SIZE         = 4
OUTPUT_DIR         = "grpo_fleet_output"

PHASE_ORDER = ["fraud_specialist", "skills_specialist", "timeline_specialist", "overseer"]

PHASE_BEST_ACTIONS = {
    "fraud_specialist":    ["verify_credential", "check_reference"],
    "skills_specialist":   ["view_section", "ask_clarification"],
    "timeline_specialist": ["view_section", "ask_clarification"],
    "overseer":            ["read_reports", "submit_final_decision"],
}

HIGH_VALUE_SECTIONS = {"experience", "education", "skills"}

SYSTEM_PROMPT = """\
You are a hiring fleet agent. Based on the observation below, output a single JSON \
action object and nothing else.

Valid action types:
  fraud_specialist  → verify_credential | check_reference | view_section | submit_specialist_report
  skills_specialist → view_section | ask_clarification | submit_specialist_report
  timeline_specialist → view_section | ask_clarification | submit_specialist_report
  overseer          → read_reports | request_reinvestigation | submit_final_decision

Examples:
  {"action_type": "verify_credential"}
  {"action_type": "check_reference", "reference_id": "ref2"}
  {"action_type": "view_section", "section": "experience"}
  {"action_type": "ask_clarification", "question": "When exactly did you join TechCorp?"}
  {"action_type": "submit_specialist_report", "findings": "Credential FAILED. Reference denies employment.", "has_issues": true, "specialist_confidence": 0.9}
  {"action_type": "read_reports", "report_target": "fraud_specialist"}
  {"action_type": "submit_final_decision", "decision": "reject", "fraud_flag": true, "confidence": 0.88, "fraud_reasoning": "Credential FAILED, reference denied employment."}

Output ONLY the JSON object. No explanation."""


# ─────────────────────────────────────────────────────────────────────────────
# Observation → prompt string
# ─────────────────────────────────────────────────────────────────────────────
def obs_to_prompt(obs: dict) -> str:
    lines = [
        f"PHASE: {obs.get('current_phase', '?')}",
        f"STEPS REMAINING: {obs.get('steps_remaining', 0)}  "
        f"(total left: {obs.get('total_steps_remaining', 0)})",
        f"VIOLATIONS SO FAR: {obs.get('violations_count', 0)}",
        "",
        f"ROLE INSTRUCTIONS:\n{obs.get('role_instructions', '')}",
        "",
        f"JOB DESCRIPTION:\n{obs.get('job_description', '')}",
    ]

    visible = obs.get("visible_sections") or {}
    if visible:
        lines.append("\nVISIBLE RESUME SECTIONS:")
        for sec, content in visible.items():
            lines.append(f"  [{sec.upper()}]\n{content}")

    reports = obs.get("specialist_reports") or []
    if reports:
        lines.append("\nSPECIALIST REPORTS:")
        for r in reports:
            lines.append(
                f"  [{r['specialist_role'].upper()}] has_issues={r['has_issues']} "
                f"confidence={r['confidence']:.2f}\n  {r['findings']}"
            )

    for key, label in [
        ("reference_response",     "REFERENCE CHECK RESULT"),
        ("verification_result",    "CREDENTIAL VERIFICATION"),
        ("clarification_response", "CLARIFICATION"),
    ]:
        val = obs.get(key)
        if val:
            lines.append(f"\n{label}:\n{val}")

    read_details = obs.get("read_report_details") or {}
    if read_details:
        lines.append("\nFULL REPORT DETAILS:")
        for role, detail in read_details.items():
            lines.append(f"  [{role.upper()}]\n{detail}")

    feedback = obs.get("feedback")
    if feedback:
        lines.append(f"\nFEEDBACK: {feedback}")

    lines.append(f"\nAVAILABLE ACTIONS: {obs.get('available_actions', [])}")
    lines.append("\nYour JSON action:")
    return "\n".join(lines)


# ─────────────────────────────────────────────────────────────────────────────
# Rule-based agent
# ─────────────────────────────────────────────────────────────────────────────
def rule_action(obs: dict) -> dict:
    phase     = obs.get("current_phase", "")
    available = obs.get("available_actions", [])

    def pick(preferred):
        for a in preferred:
            if a in available:
                return a
        return available[0] if available else "submit_specialist_report"

    if phase == "fraud_specialist":
        choice = pick(["verify_credential", "check_reference", "submit_specialist_report"])
        if choice == "check_reference":
            return {"action_type": "check_reference", "reference_id": "ref2"}
        if choice == "submit_specialist_report":
            vr  = obs.get("verification_result", "") or ""
            rr  = obs.get("reference_response",  "") or ""
            bad = "FAILED" in vr or "cannot verify" in rr.lower() or "not in our system" in rr.lower()
            return {
                "action_type": "submit_specialist_report",
                "findings": f"Verification: {vr[:120]}. Reference: {rr[:120]}.",
                "has_issues": bad,
                "specialist_confidence": 0.85 if bad else 0.7,
            }
        return {"action_type": choice}

    if phase == "skills_specialist":
        viewed = set(obs.get("visible_sections", {}).keys())
        want   = next((s for s in ["experience", "education", "skills", "projects"]
                       if s not in viewed and "view_section" in available), None)
        if want:
            return {"action_type": "view_section", "section": want}
        return {
            "action_type": "submit_specialist_report",
            "findings": "Reviewed skills sections.",
            "has_issues": False,
            "specialist_confidence": 0.7,
        }

    if phase == "timeline_specialist":
        viewed = set(obs.get("visible_sections", {}).keys())
        want   = next((s for s in ["experience", "header", "summary"]
                       if s not in viewed and "view_section" in available), None)
        if want:
            return {"action_type": "view_section", "section": want}
        return {
            "action_type": "submit_specialist_report",
            "findings": "Timeline reviewed.",
            "has_issues": False,
            "specialist_confidence": 0.65,
        }

    if phase == "overseer":
        already_read = set(obs.get("reports_read", []))
        for target in ["fraud_specialist", "skills_specialist", "timeline_specialist"]:
            if target not in already_read and "read_reports" in available:
                return {"action_type": "read_reports", "report_target": target}
        reports          = obs.get("specialist_reports", [])
        has_issues_count = sum(1 for r in reports if r.get("has_issues"))
        is_fraud         = has_issues_count >= 2
        return {
            "action_type": "submit_final_decision",
            "decision":   "reject" if is_fraud else "accept",
            "fraud_flag":  is_fraud,
            "confidence":  0.8,
            "fraud_reasoning": "Multiple specialists flagged issues." if is_fraud else "",
        }

    return {"action_type": available[0]} if available else {"action_type": "submit_specialist_report"}


# ─────────────────────────────────────────────────────────────────────────────
# Offline data collection
# ─────────────────────────────────────────────────────────────────────────────
def collect_prompts(n_episodes: int) -> list[dict]:
    collected = []
    for i in range(n_episodes):
        task  = ["easy", "medium", "hard"][i % 3]
        seed  = random.randint(0, 999)
        ep_id = f"collect-{i}-{seed}"

        try:
            resp = requests.post(
                f"{ENV_URL}/reset",
                json={"task_type": task, "seed": seed, "episode_id": ep_id},
                timeout=20,
            )
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  [collect] episode {i:3d} | reset failed: {type(e).__name__}: {e}")
            traceback.print_exc()
            continue

        obs   = data.get("observation") or data
        steps = 0

        while steps < 20:
            phase     = obs.get("current_phase", "complete")
            if phase in ("complete", None):
                break
            available = obs.get("available_actions", [])
            if not available:
                break

            collected.append({
                "prompt":            obs_to_prompt(obs),
                "phase":             phase,
                "available_actions": json.dumps(available),
                "steps_remaining":   obs.get("steps_remaining", 0),
            })

            action               = rule_action(obs)
            action["episode_id"] = ep_id

            try:
                resp = requests.post(
                    f"{ENV_URL}/step",
                    json={"action": action},
                    timeout=20,
                )
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                print(f"  [collect] episode {i:3d} step {steps} | step failed: {type(e).__name__}: {e}")
                break

            obs    = data.get("observation") or data
            steps += 1

            if data.get("done"):
                break

        if (i + 1) % 6 == 0:
            print(f"  collected {len(collected)} prompts after {i+1} episodes …")

    return collected


# ─────────────────────────────────────────────────────────────────────────────
# Reward function
# ─────────────────────────────────────────────────────────────────────────────
def score_completion(completion: str, phase: str, available_actions: list) -> float:
    reward = 0.0

    if isinstance(completion, list):
        completion = completion[0] if completion else ""
    completion = str(completion)

    action = None
    match  = re.search(r'\{[^{}]+\}', completion, re.DOTALL)
    if match:
        try:
            action = json.loads(match.group())
            reward += 0.15
        except json.JSONDecodeError:
            pass

    if action is None:
        return 0.0

    action_type = action.get("action_type", "")

    if action_type in available_actions:
        reward += 0.25
    else:
        return reward

    best = PHASE_BEST_ACTIONS.get(phase, [])
    if action_type in best[:1]:
        reward += 0.25
    elif action_type in best:
        reward += 0.10

    if action_type == "check_reference":
        if action.get("reference_id") == "ref2":
            reward += 0.20
        elif action.get("reference_id") == "ref1":
            reward += 0.10

    elif action_type == "view_section":
        section = action.get("section", "")
        if section in HIGH_VALUE_SECTIONS:
            reward += 0.20
        elif section:
            reward += 0.08

    elif action_type == "read_reports":
        if action.get("report_target") in ["fraud_specialist", "skills_specialist", "timeline_specialist"]:
            reward += 0.20

    elif action_type == "ask_clarification":
        q = action.get("question", "")
        if len(q) > 20:
            reward += 0.20
        elif q:
            reward += 0.08

    elif action_type in ("submit_specialist_report", "submit_final_decision"):
        if action_type == "submit_specialist_report":
            has_findings = bool(action.get("findings", "").strip())
            has_issues   = action.get("has_issues") is not None
            has_conf     = isinstance(action.get("specialist_confidence"), (int, float))
            reward += 0.10 if has_findings else 0.0
            reward += 0.05 if has_issues   else 0.0
            reward += 0.05 if has_conf     else 0.0
        else:
            has_decision = action.get("decision") in ("accept", "reject")
            has_flag     = action.get("fraud_flag") is not None
            has_conf     = isinstance(action.get("confidence"), (int, float))
            reward += 0.10 if has_decision else 0.0
            reward += 0.05 if has_flag     else 0.0
            reward += 0.05 if has_conf     else 0.0

    reasoning     = action.get("findings") or action.get("fraud_reasoning") or ""
    fraud_keywords = {
        "failed", "denied", "fabricated", "mismatch", "unverifiable",
        "cannot verify", "not in our system", "inflated", "exaggerated",
        "conflict", "impossible", "fabrication",
    }
    if any(kw in reasoning.lower() for kw in fraud_keywords):
        reward += 0.15
    elif len(reasoning) > 30:
        reward += 0.05

    return min(reward, 1.0)


def make_reward_fn(records: list[dict]):
    prompt_to_meta = {}
    for r in records:
        key = r["prompt"]
        prompt_to_meta[key] = {
            "phase":             r["phase"],
            "available_actions": json.loads(r["available_actions"]),
        }

    def reward_fn(completions, prompts=None, **kwargs):
        rewards = []
        for i, completion in enumerate(completions):
            if prompts is not None and i < len(prompts):
                prompt_msgs = prompts[i]
                user_text   = next(
                    (m["content"] for m in reversed(prompt_msgs) if m["role"] == "user"),
                    ""
                )
            else:
                user_text = ""

            meta              = prompt_to_meta.get(user_text, {})
            phase             = meta.get("phase", "fraud_specialist")
            available_actions = meta.get("available_actions", [])

            rewards.append(score_completion(completion, phase, available_actions))

        return rewards

    return reward_fn


# ─────────────────────────────────────────────────────────────────────────────
# Reward logger callback
# ─────────────────────────────────────────────────────────────────────────────
class RewardLoggerCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            reward = logs.get("rewards/mean")
            if reward is not None:
                print(f"  Step {state.global_step} | rewards/mean = {reward:.4f}")


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────
def main():
    global trainer, records  # expose to notebook scope
    # ── Clear old output ──────────────────────────────────────────────────
    import shutil
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
        print(f"Cleared old output dir: {OUTPUT_DIR}")

    # ── Verify environment ────────────────────────────────────────────────
    print(f"ENV_URL  : {ENV_URL}")
    print(f"MODEL    : {MODEL_NAME}")
    try:
        health = requests.get(f"{ENV_URL}/health", timeout=15).json()
        print(f"Health   : {health}\n")
    except Exception as e:
        raise RuntimeError(f"Cannot reach environment at {ENV_URL}\n{e}")

    # ── Step 1: Collect prompts ───────────────────────────────────────────
    print(f"Collecting prompts from {N_COLLECT_EPISODES} episodes …")
    records = collect_prompts(N_COLLECT_EPISODES)

    if not records:
        raise RuntimeError("No prompts collected.")
    print(f"Collected {len(records)} prompts.\n")

    # ── Step 2: Dataset ───────────────────────────────────────────────────
    dataset = Dataset.from_list([
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": r["prompt"]},
            ]
        }
        for r in records
    ])
    print(f"Dataset  : {len(dataset)} rows")

    # ── Step 3: Load model ────────────────────────────────────────────────
    print("Loading model …")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN or None,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN or None,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = model.to(device)
    print(f"Model loaded on {device}\n")

    # ── Step 4: GRPO Config ───────────────────────────────────────────────
    grpo_cfg = GRPOConfig(
        output_dir=OUTPUT_DIR,
        do_train=True,
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=4,
        max_completion_length=80,
        temperature=0.8,
        beta=0.04,
        learning_rate=2e-5,
        logging_steps=5,
        save_steps=50,
        report_to="none",
        fp16=True,
        gradient_checkpointing=True,
    )

    # ── Step 5: Reward ────────────────────────────────────────────────────
    reward_fn = make_reward_fn(records)

    # ── Step 6: Train ─────────────────────────────────────────────────────
    trainer = GRPOTrainer(
        model=model,
        args=grpo_cfg,
        train_dataset=dataset,
        reward_funcs=reward_fn,
        processing_class=tokenizer,
        callbacks=[RewardLoggerCallback()],
    )

    print(f"Training steps expected: {trainer.args.num_train_epochs * len(dataset)}")
    print("Starting GRPO training …")
    result = trainer.train()
    print("Training complete!")
    print(f"Global steps completed: {trainer.state.global_step}")

    # ── Step 7: Save ──────────────────────────────────────────────────────
    save_path = f"{OUTPUT_DIR}/final"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved to {save_path}")

    # ── Step 8: Plot ──────────────────────────────────────────────────────
    try:
        import matplotlib.pyplot as plt

        history = trainer.state.log_history
        steps   = [h["step"]         for h in history if "rewards/mean" in h]
        rewards = [h["rewards/mean"] for h in history if "rewards/mean" in h]

        if steps:
            os.makedirs("assets", exist_ok=True)
            plt.figure(figsize=(10, 4))
            plt.plot(steps, rewards)
            plt.xlabel("Steps")
            plt.ylabel("Reward")
            plt.title("Training Reward Curve")
            plt.grid()
            plt.savefig("assets/reward_curve.png", dpi=150)
            plt.show()
            print("Saved reward curve!")
        else:
            print("No reward data to plot.")
            print("Full log history:")
            for h in history:
                print(h)
    except Exception as e:
        print("Plot failed:", e)


if __name__ == "__main__":
    main()

Cleared old output dir: grpo_fleet_output
ENV_URL  : https://ishikamahadar-resume-env.hf.space
MODEL    : Qwen/Qwen2.5-1.5B-Instruct
Health   : {'status': 'healthy'}

  collected 66 prompts after 6 episodes …
  collected 132 prompts after 12 episodes …
  collected 198 prompts after 18 episodes …
  collected 264 prompts after 24 episodes …
  collected 330 prompts after 30 episodes …
  collected 396 prompts after 36 episodes …
Collected 396 prompts.

Dataset  : 396 rows
Loading model …


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
Model loaded on cuda



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Training steps expected: 792
Starting GRPO training …


Step,Training Loss
5,0.150582
10,0.018786
15,0.044773
20,0.000054
25,-0.152353
30,-0.087150
35,0.000077
40,-0.034363
45,0.012574
50,-0.061733


Training complete!
Global steps completed: 792
Saved to grpo_fleet_output/final
No reward data to plot.
Full log history:
{'loss': 0.1505819797515869, 'grad_norm': 1.6746939420700073, 'learning_rate': 1.98989898989899e-05, 'num_tokens': 14859.0, 'completions/mean_length': 15.55, 'completions/min_length': 13.8, 'completions/max_length': 20.8, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 15.55, 'completions/min_terminated_length': 13.8, 'completions/max_terminated_length': 20.8, 'rewards/reward_fn/mean': 0.735500019788742, 'rewards/reward_fn/std': 0.09085640460252761, 'reward': 0.735500019788742, 'reward_std': 0.09085640460252761, 'frac_reward_zero_std': 0.4, 'kl': 6.388259484992886e-05, 'entropy': 0.05571887968108058, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'step_time': 8.091224659600083, 'epoch': 0.012626262626262626, 'step': 5}
{'loss': 0.01878616213798523,

In [ ]:
main()

In [ ]:
import matplotlib.pyplot as plt
import json

# ── Extract reward data from log history ──────────────────────────────────
history = trainer.state.log_history

steps        = [h["step"]                  for h in history if "rewards/reward_fn/mean" in h]
rewards_mean = [h["rewards/reward_fn/mean"] for h in history if "rewards/reward_fn/mean" in h]
rewards_std  = [h["rewards/reward_fn/std"]  for h in history if "rewards/reward_fn/std"  in h]
kl_vals      = [h["kl"]                    for h in history if "kl"                      in h]
loss_vals    = [h["loss"]                  for h in history if "loss"                    in h]
loss_steps   = [h["step"]                  for h in history if "loss"                    in h]

os.makedirs("assets", exist_ok=True)

# ── Plot 1: Reward mean + std band ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, rewards_mean, label="rewards/mean", color="steelblue", linewidth=1.5)
ax.fill_between(
    steps,
    [r - s for r, s in zip(rewards_mean, rewards_std)],
    [r + s for r, s in zip(rewards_mean, rewards_std)],
    alpha=0.2, color="steelblue", label="±1 std"
)
ax.set_xlabel("Step")
ax.set_ylabel("Reward")
ax.set_title("GRPO Training — Reward Curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("assets/reward_curve.png", dpi=150)
plt.show()
print("Saved reward_curve.png")

# ── Plot 2: KL divergence ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(steps, kl_vals, color="darkorange", linewidth=1.2)
ax.set_xlabel("Step")
ax.set_ylabel("KL Divergence")
ax.set_title("KL Divergence from Reference Policy")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("assets/kl_curve.png", dpi=150)
plt.show()
print("Saved kl_curve.png")

# ── Plot 3: Training loss ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(loss_steps, loss_vals, color="crimson", linewidth=1.0, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("GRPO Policy Loss (oscillation around 0 is normal)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("assets/loss_curve.png", dpi=150)
plt.show()
print("Saved loss_curve.png")

# ── Save log history as JSON for later ───────────────────────────────────
with open("assets/log_history.json", "w") as f:
    json.dump(history, f, indent=2)
print("Saved log_history.json")

print(f"\nFinal reward: {rewards_mean[-1]:.4f}")
print(f"Best reward:  {max(rewards_mean):.4f} at step {steps[rewards_mean.index(max(rewards_mean))]}")
print(f"Start reward: {rewards_mean[0]:.4f}")
print(f"Improvement:  +{rewards_mean[-1] - rewards_mean[0]:.4f}")

In [8]:
!ls /content/grpo_fleet_output/final/

adapter_config.json	   README.md		  tokenizer.json
adapter_model.safetensors  ref			  training_args.bin
chat_template.jinja	   tokenizer_config.json


In [15]:
from google.colab import files
import zipfile

# Zip everything into one file
with zipfile.ZipFile("grpo_results.zip", "w") as zf:
    # Add model files
    for root, dirs, fs in os.walk("grpo_fleet_output/final"):
        for f in fs:
            zf.write(os.path.join(root, f))
    # Add plots and logs
    for root, dirs, fs in os.walk("assets"):
        for f in fs:
            zf.write(os.path.join(root, f))

# This triggers a download in your browser
files.download("grpo_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>